# Пример 01: Брзи разговор преко OpenAI SDK-а

Овај нотебук демонстрира како користити OpenAI SDK са Microsoft Foundry Local за брзе интеракције у разговору.

## Преглед

Овај пример показује:
- Коришћење OpenAI Python SDK-а са Foundry Local
- Руковање конфигурацијама за Azure OpenAI и локални Foundry
- Имплементацију правилног руковања грешкама и стратегија за резервне опције
- Коришћење FoundryLocalManager-а за управљање услугама


## Предуслови

Уверите се да имате:
1. Инсталиран и покренут Foundry Local
2. Учитан модел (нпр. `foundry model run phi-4-mini`)
3. Инсталиране потребне Python библиотеке

### Инсталирање зависности


In [ ]:
# Install required packages if not already installed
!pip install openai foundry-local-sdk

## Импортовање потребних библиотека


In [ ]:
import os
import sys
from openai import OpenAI

try:
    from foundry_local import FoundryLocalManager
    FOUNDRY_SDK_AVAILABLE = True
    print("✅ Foundry Local SDK is available")
except ImportError:
    FOUNDRY_SDK_AVAILABLE = False
    print("⚠️ Foundry Local SDK not available, will use manual configuration")

## Конфигурација

Подесите своју конфигурацију. Можете користити једну од следећих опција:
1. **Azure OpenAI** (засновано на облаку)
2. **Foundry Local** (локално закључивање)

### Опција 1: Конфигурација Azure OpenAI


In [ ]:
# Azure OpenAI Configuration (uncomment and set your values)
# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-resource.openai.azure.com"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-api-key"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-08-01-preview"
# os.environ["MODEL"] = "your-deployment-name"

### Опција 2: Локална конфигурација Foundry


In [ ]:
# Foundry Local Configuration
MODEL_ALIAS = "phi-4-mini"  # Change this to your preferred model
BASE_URL = "http://localhost:8000"  # Default Foundry Local URL
API_KEY = ""  # Usually not needed for local

## Иницијализација клијента

Креирајте OpenAI клијента на основу ваше конфигурације:


In [ ]:
def create_client():
    """Create and configure OpenAI client."""
    
    # Check for Azure OpenAI configuration
    azure_endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
    azure_api_key = os.environ.get("AZURE_OPENAI_API_KEY")
    azure_api_version = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    
    if azure_endpoint and azure_api_key:
        # Azure OpenAI path
        model = os.environ.get("MODEL", "your-deployment-name")
        client = OpenAI(
            base_url=f"{azure_endpoint}/openai",
            api_key=azure_api_key,
            default_query={"api-version": azure_api_version},
        )
        print(f"🌐 Using Azure OpenAI with model: {model}")
        return client, model
    
    # Foundry Local path
    if FOUNDRY_SDK_AVAILABLE:
        try:
            # Use FoundryLocalManager for proper service management
            manager = FoundryLocalManager(MODEL_ALIAS)
            model_info = manager.get_model_info(MODEL_ALIAS)
            
            # Configure OpenAI client to use local Foundry service
            client = OpenAI(
                base_url=manager.endpoint,
                api_key=manager.api_key  # API key is not required for local usage
            )
            model = model_info.id
            print(f"🏠 Using Foundry Local SDK with model: {model}")
            return client, model
        except Exception as e:
            print(f"⚠️ Could not use Foundry SDK ({e}), falling back to manual configuration")
    
    # Fallback to manual configuration
    client = OpenAI(
        base_url=f"{BASE_URL}/v1",
        api_key=API_KEY
    )
    model = MODEL_ALIAS
    print(f"🔧 Using manual configuration with model: {model}")
    return client, model

# Initialize the client
client, model = create_client()

## Основни пример за ћаскање

Хајде да тестирамо једноставну интеракцију у ћаскању:


In [ ]:
def chat_with_model(prompt, max_tokens=128):
    """Send a chat message to the model and get a response."""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test with a simple prompt
prompt = "Say hello from Foundry Local and explain what you are in one sentence."
print(f"👤 User: {prompt}")
print("\n🤖 Assistant:")
response = chat_with_model(prompt)
print(response)

## Интерактивна сесија чета

Пробајте различите типове упита да видите како модел реагује:


In [ ]:
# Example prompts to try
example_prompts = [
    "What are the benefits of edge AI?",
    "Explain the difference between local and cloud AI inference.",
    "Write a short Python function to calculate the factorial of a number.",
    "What is Microsoft Foundry Local?"
]

for i, prompt in enumerate(example_prompts, 1):
    print(f"\n{'='*50}")
    print(f"Example {i}: {prompt}")
    print(f"{'='*50}")
    
    response = chat_with_model(prompt, max_tokens=200)
    print(response)

## Напредна употреба: Стримовање одговора

За дуже одговоре, стримовање може пружити боље корисничко искуство:


In [ ]:
def chat_with_streaming(prompt, max_tokens=300):
    """Send a chat message with streaming response."""
    try:
        stream = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            stream=True
        )
        
        print("🤖 Assistant (streaming):")
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                content = chunk.choices[0].delta.content
                print(content, end="", flush=True)
                full_response += content
        print("\n")  # New line after streaming
        return full_response
    except Exception as e:
        return f"Error: {e}"

# Test streaming with a longer prompt
streaming_prompt = "Explain the advantages of running AI models locally on device versus in the cloud. Include privacy, latency, and cost considerations."
print(f"👤 User: {streaming_prompt}\n")
streaming_response = chat_with_streaming(streaming_prompt)

## Обрада грешака и дијагностика

Хајде да проверимо статус услуге и доступне моделе:


In [ ]:
def check_service_health():
    """Check Foundry Local service health."""
    try:
        # Try to list available models
        models_response = client.models.list()
        models = [model.id for model in models_response.data]
        
        print("✅ Service is healthy")
        print(f"📋 Available models: {models}")
        print(f"🎯 Current model: {model}")
        
        if model in models:
            print("✅ Current model is available")
        else:
            print("⚠️ Current model may not be loaded")
            
    except Exception as e:
        print(f"❌ Service check failed: {e}")
        print("\n🔧 Troubleshooting tips:")
        print("1. Make sure Foundry Local is running")
        print("2. Check if a model is loaded: foundry service ps")
        print("3. Verify the endpoint URL is correct")

check_service_health()

## Тестирање прилагођених упита

Користите ћелију испод за тестирање сопствених упита:


In [ ]:
# Enter your custom prompt here
custom_prompt = "Write a haiku about artificial intelligence running on edge devices."

print(f"👤 User: {custom_prompt}\n")
custom_response = chat_with_model(custom_prompt, max_tokens=100)
print(f"🤖 Assistant: {custom_response}")

## Резиме

Овај нотебук је показао:

1. **✅ Клијентска конфигурација**: Како конфигурисати OpenAI SDK са Foundry Local
2. **✅ Основни разговор**: Једноставне интеракције захтев-одговор
3. **✅ Стриминг**: Генерисање одговора у реалном времену
4. **✅ Руковање грешкама**: Робусно руковање грешкама и дијагностика
5. **✅ Здравље сервиса**: Провера доступности модела и статуса сервиса

### Следећи кораци

- **Пример 02**: Напредна интеграција SDK-а са подршком за Azure OpenAI
- **Пример 04**: Изградња Chainlit апликације за разговор
- **Пример 05**: Системи за оркестрацију више агената
- **Пример 06**: Интелигентно усмеравање модела

### Кључне предности Foundry Local

- 🔒 **Приватност**: Подаци никада не напуштају ваш уређај
- ⚡ **Брзина**: Локална инференција са малим кашњењем
- 💰 **Трошкови**: Нема трошкова коришћења API-ја
- 🔌 **Офлајн**: Ради без интернет конекције
- 🛠️ **Компатибилност**: API компатибилан са OpenAI
